In [1]:
# Cell 2A — Build all-tenor Corsi call candidate panel
# Purpose:
#   Build the raw all-tenor candidate panel for the Corsi-based short-call sleeve.
#
# Scope:
#   - Research only
#   - All existing tenors: 9, 12, 15, 18, 21, 24, 27, 30, 33
#   - Uses existing SPX-derived VIX-style vol and Corsi VRP inputs
#   - Builds raw theoretical 1SD / 3SD call strikes using SPY close
#   - No ThetaData option pricing yet
#   - No parameter sweep yet
#   - No selected trade logic yet
#
# Trade structure:
#   - Sell 1SD call
#   - Buy 3SD call
#   - SD = vix_style_vol_pct / 100 * sqrt(tenor / 365)
#
# Signal model to be researched later:
#   - Corsi VRP = ln(implied_variance / forecast_variance)
#   - Use model_vrp_log_final, z_3m_final, z_1y_final
#   - During initial sweeps, 3m z threshold = 1y z threshold

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FINAL_SIGNAL_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "vrp_final_signal"
    / "vrp_final_corsi_signal_base_panel_v1.parquet"
)

OUTPUT_PANEL_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet"
)

print("=" * 100)
print("Cell 2A — Build all-tenor Corsi call candidate panel")
print("=" * 100)
print(f"Input:   {FINAL_SIGNAL_PATH}")
print(f"Output:  {OUTPUT_PANEL_PATH}")
print()

# --------------------------------------------------------------------------------------
# Parameters
# --------------------------------------------------------------------------------------

EXPECTED_TENORS = [9, 12, 15, 18, 21, 24, 27, 30, 33]

# --------------------------------------------------------------------------------------
# Load source
# --------------------------------------------------------------------------------------

df = pd.read_parquet(FINAL_SIGNAL_PATH)

required_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "spy_close",
    "implied_variance_final",
    "vix_style_vol_final",
    "forecast_variance_final",
    "forecast_vol_final",
    "model_vrp_log_final",
    "z_3m_final",
    "z_1y_final",
    "rsi14_final",
]

missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns from final signal panel: {missing_cols}")

# --------------------------------------------------------------------------------------
# Build candidate panel
# --------------------------------------------------------------------------------------

panel = df.loc[df["tenor"].isin(EXPECTED_TENORS), required_cols].copy()

panel = panel.rename(
    columns={
        "trade_date": "trade_date",
        "tenor": "tenor",
        "tenor_bucket": "tenor_bucket",
        "spy_close": "spy_close",
        "implied_variance_final": "implied_variance",
        "vix_style_vol_final": "vix_style_vol_pct",
        "forecast_variance_final": "forecast_variance",
        "forecast_vol_final": "forecast_vol_pct",
        "model_vrp_log_final": "corsi_vrp_log",
        "z_3m_final": "corsi_vrp_z_3m",
        "z_1y_final": "corsi_vrp_z_1y",
        "rsi14_final": "rsi14",
    }
)

panel["trade_date"] = pd.to_datetime(panel["trade_date"], errors="coerce")
panel = panel.sort_values(["trade_date", "tenor"]).reset_index(drop=True)

# --------------------------------------------------------------------------------------
# Guardrails
# --------------------------------------------------------------------------------------

if panel.empty:
    raise ValueError("All-tenor Corsi call candidate panel is empty.")

duplicate_date_tenor = panel.duplicated(["trade_date", "tenor"]).sum()
if duplicate_date_tenor > 0:
    raise ValueError(f"Duplicate trade_date × tenor rows found: {duplicate_date_tenor:,}")

actual_tenors = sorted(panel["tenor"].dropna().unique().tolist())
missing_tenors = sorted(set(EXPECTED_TENORS) - set(actual_tenors))
unexpected_tenors = sorted(set(actual_tenors) - set(EXPECTED_TENORS))

if missing_tenors:
    raise ValueError(f"Missing expected tenors: {missing_tenors}")

if unexpected_tenors:
    raise ValueError(f"Unexpected tenors found: {unexpected_tenors}")

vix_median = panel["vix_style_vol_pct"].dropna().median()
forecast_vol_median = panel["forecast_vol_pct"].dropna().median()

if pd.notna(vix_median) and vix_median <= 1.0:
    raise ValueError(
        "vix_style_vol_pct appears to be decimal format, not percent format. "
        f"Median value is {vix_median:.6f}. Expected values like 12, 18, 25."
    )

if pd.notna(forecast_vol_median) and forecast_vol_median <= 1.0:
    raise ValueError(
        "forecast_vol_pct appears to be decimal format, not percent format. "
        f"Median value is {forecast_vol_median:.6f}. Expected values like 8, 14, 25."
    )

# --------------------------------------------------------------------------------------
# Raw theoretical strike construction
# --------------------------------------------------------------------------------------

panel["sqrt_t"] = np.sqrt(panel["tenor"] / 365.0)
panel["sd_move_decimal"] = (panel["vix_style_vol_pct"] / 100.0) * panel["sqrt_t"]

panel["short_call_1sd_raw"] = panel["spy_close"] * (1.0 + panel["sd_move_decimal"])
panel["long_call_3sd_raw"] = panel["spy_close"] * (1.0 + 3.0 * panel["sd_move_decimal"])

panel["call_spread_width_raw"] = (
    panel["long_call_3sd_raw"] - panel["short_call_1sd_raw"]
)

panel["short_call_otm_pct_raw"] = (
    panel["short_call_1sd_raw"] / panel["spy_close"] - 1.0
)

panel["long_call_otm_pct_raw"] = (
    panel["long_call_3sd_raw"] / panel["spy_close"] - 1.0
)

# Useful selection/sweep helper fields, but no selection decision yet.
panel["avg_corsi_z"] = (
    panel["corsi_vrp_z_3m"] + panel["corsi_vrp_z_1y"]
) / 2.0

panel["min_corsi_z"] = panel[["corsi_vrp_z_3m", "corsi_vrp_z_1y"]].min(axis=1)

# --------------------------------------------------------------------------------------
# Missingness and coverage audits
# --------------------------------------------------------------------------------------

core_cols = [
    "trade_date",
    "tenor",
    "tenor_bucket",
    "spy_close",
    "implied_variance",
    "vix_style_vol_pct",
    "forecast_variance",
    "forecast_vol_pct",
    "corsi_vrp_log",
    "corsi_vrp_z_3m",
    "corsi_vrp_z_1y",
    "rsi14",
    "short_call_1sd_raw",
    "long_call_3sd_raw",
    "call_spread_width_raw",
]

missing_summary = (
    panel[core_cols]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary["missing_pct"] = missing_summary["missing_count"] / len(panel)

panel["candidate_inputs_complete"] = panel[core_cols].notna().all(axis=1)

tenor_summary = (
    panel
    .groupby("tenor", as_index=False)
    .agg(
        rows=("trade_date", "count"),
        date_min=("trade_date", "min"),
        date_max=("trade_date", "max"),
        complete_rows=("candidate_inputs_complete", "sum"),
        median_vix_style_vol_pct=("vix_style_vol_pct", "median"),
        median_forecast_vol_pct=("forecast_vol_pct", "median"),
        median_corsi_vrp_log=("corsi_vrp_log", "median"),
        median_z_3m=("corsi_vrp_z_3m", "median"),
        median_z_1y=("corsi_vrp_z_1y", "median"),
        median_rsi14=("rsi14", "median"),
        median_short_call_otm_pct_raw=("short_call_otm_pct_raw", "median"),
        median_long_call_otm_pct_raw=("long_call_otm_pct_raw", "median"),
        median_call_spread_width_raw=("call_spread_width_raw", "median"),
    )
)

tenor_summary["complete_row_pct"] = tenor_summary["complete_rows"] / tenor_summary["rows"]

date_coverage = (
    panel
    .groupby("trade_date", as_index=False)
    .agg(
        tenor_count=("tenor", "nunique"),
        complete_candidate_count=("candidate_inputs_complete", "sum"),
    )
)

date_coverage["has_all_expected_tenors"] = date_coverage["tenor_count"] == len(EXPECTED_TENORS)
date_coverage["has_all_complete_candidates"] = (
    date_coverage["complete_candidate_count"] == len(EXPECTED_TENORS)
)

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

audit_panel_path = AUDIT_DIR / f"02A_call_sleeve_corsi_all_tenor_raw_candidate_panel_{timestamp}.csv"
missing_summary_path = AUDIT_DIR / f"02A_call_sleeve_corsi_missing_summary_{timestamp}.csv"
tenor_summary_path = AUDIT_DIR / f"02A_call_sleeve_corsi_tenor_summary_{timestamp}.csv"
date_coverage_path = AUDIT_DIR / f"02A_call_sleeve_corsi_date_coverage_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02A_call_sleeve_corsi_raw_candidate_manifest_{timestamp}.json"

panel.to_parquet(OUTPUT_PANEL_PATH, index=False)
panel.to_csv(audit_panel_path, index=False)
missing_summary.to_csv(missing_summary_path, index=False)
tenor_summary.to_csv(tenor_summary_path, index=False)
date_coverage.to_csv(date_coverage_path, index=False)

manifest = {
    "cell": "2A",
    "description": "All-tenor Corsi short-call raw candidate panel",
    "created_at": timestamp,
    "input_path": str(FINAL_SIGNAL_PATH),
    "output_panel_path": str(OUTPUT_PANEL_PATH),
    "audit_panel_path": str(audit_panel_path),
    "missing_summary_path": str(missing_summary_path),
    "tenor_summary_path": str(tenor_summary_path),
    "date_coverage_path": str(date_coverage_path),
    "expected_tenors": EXPECTED_TENORS,
    "actual_tenors": actual_tenors,
    "date_min": str(panel["trade_date"].min().date()),
    "date_max": str(panel["trade_date"].max().date()),
    "rows": int(len(panel)),
    "unique_dates": int(panel["trade_date"].nunique()),
    "duplicate_date_tenor_rows": int(duplicate_date_tenor),
    "complete_candidate_rows": int(panel["candidate_inputs_complete"].sum()),
    "complete_candidate_pct": float(panel["candidate_inputs_complete"].mean()),
    "date_rows_with_all_expected_tenors": int(date_coverage["has_all_expected_tenors"].sum()),
    "date_rows_with_all_complete_candidates": int(date_coverage["has_all_complete_candidates"].sum()),
    "sqrt_t_convention": "sqrt(tenor / 365)",
    "short_call_raw_formula": "spy_close * (1 + vix_style_vol_pct / 100 * sqrt(tenor / 365))",
    "long_call_raw_formula": "spy_close * (1 + 3 * vix_style_vol_pct / 100 * sqrt(tenor / 365))",
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("Built all-tenor Corsi call candidate panel")
print(f"Rows:                              {len(panel):,}")
print(f"Unique dates:                      {panel['trade_date'].nunique():,}")
print(f"Date range:                        {panel['trade_date'].min().date()} to {panel['trade_date'].max().date()}")
print(f"Tenors:                            {actual_tenors}")
print(f"Duplicate date × tenor rows:       {duplicate_date_tenor:,}")
print(f"Complete candidate rows:           {int(panel['candidate_inputs_complete'].sum()):,}")
print(f"Complete candidate pct:            {panel['candidate_inputs_complete'].mean():.2%}")
print()
print("Strike convention:")
print("  sqrt_t = sqrt(tenor / 365)")
print("  short_call_1sd_raw = spy_close * (1 + 1 * vix_style_vol_pct / 100 * sqrt_t)")
print("  long_call_3sd_raw  = spy_close * (1 + 3 * vix_style_vol_pct / 100 * sqrt_t)")
print()
print("Sanity checks:")
print(f"  Median vix_style_vol_pct:         {vix_median:.4f}")
print(f"  Median forecast_vol_pct:          {forecast_vol_median:.4f}")
print(f"  Dates with all expected tenors:   {int(date_coverage['has_all_expected_tenors'].sum()):,} / {len(date_coverage):,}")
print(f"  Dates with all complete rows:     {int(date_coverage['has_all_complete_candidates'].sum()):,} / {len(date_coverage):,}")
print()
print("Saved:")
print(f"  Output parquet:                   {OUTPUT_PANEL_PATH}")
print(f"  Audit panel CSV:                  {audit_panel_path}")
print(f"  Missing summary CSV:              {missing_summary_path}")
print(f"  Tenor summary CSV:                {tenor_summary_path}")
print(f"  Date coverage CSV:                {date_coverage_path}")
print(f"  Manifest JSON:                    {manifest_path}")

print()
print("=" * 100)
print("Missing summary")
print("=" * 100)
display(missing_summary)

print()
print("=" * 100)
print("Tenor summary")
print("=" * 100)
display(tenor_summary)

print()
print("=" * 100)
print("Latest date candidates")
print("=" * 100)
latest_date = panel["trade_date"].max()
display(
    panel.loc[panel["trade_date"] == latest_date]
    .sort_values("tenor")
    [
        [
            "trade_date",
            "tenor",
            "tenor_bucket",
            "spy_close",
            "vix_style_vol_pct",
            "forecast_vol_pct",
            "corsi_vrp_log",
            "corsi_vrp_z_3m",
            "corsi_vrp_z_1y",
            "rsi14",
            "short_call_1sd_raw",
            "long_call_3sd_raw",
            "call_spread_width_raw",
            "candidate_inputs_complete",
        ]
    ]
)

Cell 2A — Build all-tenor Corsi call candidate panel
Input:   C:\Users\patri\vrp_project\data\processed\vrp_final_signal\vrp_final_corsi_signal_base_panel_v1.parquet
Output:  C:\Users\patri\vrp_project\data\processed\call_sleeve_research\call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet

Built all-tenor Corsi call candidate panel
Rows:                              14,715
Unique dates:                      1,635
Date range:                        2020-01-02 to 2026-07-08
Tenors:                            [9, 12, 15, 18, 21, 24, 27, 30, 33]
Duplicate date × tenor rows:       0
Complete candidate rows:           12,420
Complete candidate pct:            84.40%

Strike convention:
  sqrt_t = sqrt(tenor / 365)
  short_call_1sd_raw = spy_close * (1 + 1 * vix_style_vol_pct / 100 * sqrt_t)
  long_call_3sd_raw  = spy_close * (1 + 3 * vix_style_vol_pct / 100 * sqrt_t)

Sanity checks:
  Median vix_style_vol_pct:         18.4128
  Median forecast_vol_pct:          13.9888
  Dates with al

,column,missing_count,missing_pct
0,trade_date,0,0.000000
1,tenor,0,0.000000
2,tenor_bucket,27,0.001835
3,spy_close,27,0.001835
4,implied_variance,0,0.000000
5,vix_style_vol_pct,0,0.000000
6,forecast_variance,0,0.000000
7,forecast_vol_pct,0,0.000000
8,corsi_vrp_log,0,0.000000
9,corsi_vrp_z_3m,567,0.038532



Tenor summary


,tenor,rows,date_min,date_max,complete_rows,median_vix_style_vol_pct,median_forecast_vol_pct,median_corsi_vrp_log,median_z_3m,median_z_1y,median_rsi14,median_short_call_otm_pct_raw,median_long_call_otm_pct_raw,median_call_spread_width_raw,complete_row_pct
0,9,1635,2020-01-02,2026-07-08,1380,17.515786,13.224036,0.540406,-0.042942,-0.153739,57.195255,0.027534,0.082602,26.165654,0.844037
1,12,1635,2020-01-02,2026-07-08,1380,17.809555,13.192627,0.564562,-0.041767,-0.145995,57.195255,0.032343,0.097029,30.728249,0.844037
2,15,1635,2020-01-02,2026-07-08,1380,17.917531,13.778479,0.499975,-0.067391,-0.177360,57.195255,0.036426,0.109279,34.794209,0.844037
3,18,1635,2020-01-02,2026-07-08,1380,18.212769,13.715855,0.522889,-0.060154,-0.210078,57.195255,0.040473,0.121419,38.699576,0.844037
4,21,1635,2020-01-02,2026-07-08,1380,18.384014,14.054020,0.498948,-0.068424,-0.242724,57.195255,0.044123,0.132368,42.294111,0.844037
5,24,1635,2020-01-02,2026-07-08,1380,18.580106,14.087548,0.512516,-0.093706,-0.253646,57.195255,0.047659,0.142976,45.520278,0.844037
6,27,1635,2020-01-02,2026-07-08,1380,18.750958,14.236235,0.518564,-0.117991,-0.234771,57.195255,0.051085,0.153255,48.83928,0.844037
7,30,1635,2020-01-02,2026-07-08,1380,18.950384,14.441723,0.501646,-0.140280,-0.229082,57.195255,0.054371,0.163112,51.952965,0.844037
8,33,1635,2020-01-02,2026-07-08,1380,19.104130,14.449255,0.514078,-0.148171,-0.238308,57.195255,0.057456,0.172367,54.956408,0.844037



Latest date candidates


,trade_date,tenor,tenor_bucket,spy_close,vix_style_vol_pct,forecast_vol_pct,corsi_vrp_log,corsi_vrp_z_3m,corsi_vrp_z_1y,rsi14,short_call_1sd_raw,long_call_3sd_raw,call_spread_width_raw,candidate_inputs_complete
14706,2026-07-08,9,None,NaN,14.142112,11.315075,0.446042,0.294892,-0.387636,47.0441,<NA>,<NA>,<NA>,False
14707,2026-07-08,12,None,NaN,14.623586,11.416124,0.495218,0.322649,-0.383688,47.0441,<NA>,<NA>,<NA>,False
14708,2026-07-08,15,None,NaN,14.905007,12.018682,0.430470,0.292371,-0.407554,47.0441,<NA>,<NA>,<NA>,False
14709,2026-07-08,18,None,NaN,15.478564,11.963703,0.515158,0.457871,-0.270658,47.0441,<NA>,<NA>,<NA>,False
14710,2026-07-08,21,None,NaN,16.035208,12.209146,0.545203,0.608556,-0.130088,47.0441,<NA>,<NA>,<NA>,False
14711,2026-07-08,24,None,NaN,16.397687,12.266581,0.580523,0.691769,-0.065768,47.0441,<NA>,<NA>,<NA>,False
14712,2026-07-08,27,None,NaN,16.599352,12.330703,0.594543,0.711117,-0.058179,47.0441,<NA>,<NA>,<NA>,False
14713,2026-07-08,30,None,NaN,16.758938,12.532342,0.581238,0.723058,-0.056647,47.0441,<NA>,<NA>,<NA>,False
14714,2026-07-08,33,None,NaN,16.945993,12.561058,0.598860,0.712617,-0.058700,47.0441,<NA>,<NA>,<NA>,False


In [ ]:
# Cell 2B — ThetaData option price source inventory for SPY call spreads
# Purpose:
#   Find existing local ThetaData / option-chain files and scripts that can be used
#   to attach actual SPY option bid/ask entry prices to the Corsi call candidate panel.
#
# Scope:
#   - Inventory only
#   - No ThetaData API pulls yet
#   - No option pricing yet
#   - No parameter sweep yet
#
# Looking for:
#   - SPY option chain / quote files
#   - ThetaData pull scripts
#   - Existing option pricing / backtest artifacts from prior put work
#   - Columns for date, expiration, strike, right/call-put, bid, ask, mid, quote timestamp

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime
import os
import re

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Cell 2B — ThetaData option price source inventory")
print("=" * 100)
print(f"Project root: {PROJECT_ROOT}")
print(f"Audit dir:    {AUDIT_DIR}")
print()

# --------------------------------------------------------------------------------------
# Search configuration
# --------------------------------------------------------------------------------------

SEARCH_ROOTS = [
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "notebooks",
    PROJECT_ROOT / "scripts",
]

SEARCH_ROOTS = [p for p in SEARCH_ROOTS if p.exists()]

DATA_EXTENSIONS = {".parquet", ".csv", ".feather", ".pkl", ".pickle"}
TEXT_EXTENSIONS = {".py", ".ipynb", ".txt", ".md", ".json"}

# Filename/path terms that suggest options / ThetaData relevance
PATH_KEYWORDS = [
    "theta",
    "thetadata",
    "option",
    "options",
    "chain",
    "quote",
    "quotes",
    "contract",
    "expiry",
    "expiration",
    "strike",
    "bid",
    "ask",
    "mid",
    "spy",
    "spx",
    "put",
    "call",
]

# Column terms required for actual option pricing
COLUMN_GROUPS = {
    "date": ["date", "trade_date", "quote_date", "asof_date", "timestamp", "datetime", "ms_of_day"],
    "expiration": ["expiration", "expiry", "exp", "expiration_date"],
    "strike": ["strike"],
    "right": ["right", "option_type", "cp", "call_put", "put_call", "root"],
    "bid": ["bid", "bid_price"],
    "ask": ["ask", "ask_price"],
    "mid": ["mid", "mark", "price", "close"],
    "underlying": ["underlying", "root", "symbol", "ticker"],
}

TEXT_SEARCH_TERMS = [
    "thetadata",
    "ThetaData",
    "option chain",
    "bulk_hist",
    "hist/option",
    "bid",
    "ask",
    "strike",
    "expiration",
    "SPY",
    "SPX",
]

MAX_FILE_SIZE_TO_TEXT_SCAN_MB = 10
MAX_CSV_ROWS_TO_SAMPLE = 5
MAX_DISPLAY_ROWS = 200

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def path_score(path: Path) -> int:
    s = str(path).lower()
    return sum(1 for k in PATH_KEYWORDS if k in s)

def safe_file_size_mb(path: Path) -> float:
    try:
        return path.stat().st_size / (1024 * 1024)
    except Exception:
        return np.nan

def detect_column_groups(columns):
    cols = [str(c) for c in columns]
    lower_map = {c: c.lower() for c in cols}
    
    detected = {}
    for group, terms in COLUMN_GROUPS.items():
        matches = []
        for col, low in lower_map.items():
            if any(term in low for term in terms):
                matches.append(col)
        detected[group] = matches
    
    return detected

def option_pricing_column_score(detected_groups) -> int:
    required_groups = ["date", "expiration", "strike", "right", "bid", "ask"]
    return sum(1 for g in required_groups if len(detected_groups.get(g, [])) > 0)

def read_parquet_metadata(path: Path):
    out = {
        "rows": None,
        "cols": None,
        "columns": [],
        "error": None,
    }
    
    try:
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        out["rows"] = pf.metadata.num_rows
        out["cols"] = len(pf.schema.names)
        out["columns"] = list(pf.schema.names)
    except Exception as e1:
        try:
            sample = pd.read_parquet(path)
            out["rows"] = len(sample)
            out["cols"] = len(sample.columns)
            out["columns"] = list(sample.columns)
        except Exception as e2:
            out["error"] = f"pyarrow_error={repr(e1)} | pandas_error={repr(e2)}"
    
    return out

def read_csv_metadata(path: Path):
    out = {
        "rows": None,
        "cols": None,
        "columns": [],
        "error": None,
    }
    
    try:
        sample = pd.read_csv(path, nrows=MAX_CSV_ROWS_TO_SAMPLE)
        out["cols"] = len(sample.columns)
        out["columns"] = list(sample.columns)
        # Row count for huge CSVs can be expensive; skip exact rows.
        out["rows"] = None
    except Exception as e:
        out["error"] = repr(e)
    
    return out

def read_feather_metadata(path: Path):
    out = {
        "rows": None,
        "cols": None,
        "columns": [],
        "error": None,
    }
    
    try:
        sample = pd.read_feather(path)
        out["rows"] = len(sample)
        out["cols"] = len(sample.columns)
        out["columns"] = list(sample.columns)
    except Exception as e:
        out["error"] = repr(e)
    
    return out

def read_pickle_metadata(path: Path):
    out = {
        "rows": None,
        "cols": None,
        "columns": [],
        "error": None,
    }
    
    try:
        obj = pd.read_pickle(path)
        if isinstance(obj, pd.DataFrame):
            out["rows"] = len(obj)
            out["cols"] = len(obj.columns)
            out["columns"] = list(obj.columns)
        else:
            out["error"] = f"pickle object is {type(obj)} not DataFrame"
    except Exception as e:
        out["error"] = repr(e)
    
    return out

def scan_text_file(path: Path):
    result = {
        "text_hits": [],
        "text_hit_count": 0,
        "error": None,
    }
    
    size_mb = safe_file_size_mb(path)
    if pd.notna(size_mb) and size_mb > MAX_FILE_SIZE_TO_TEXT_SCAN_MB:
        result["error"] = f"skipped_text_scan_file_too_large_{size_mb:.2f}mb"
        return result
    
    try:
        text = path.read_text(errors="ignore")
        hits = []
        for term in TEXT_SEARCH_TERMS:
            if term.lower() in text.lower():
                hits.append(term)
        result["text_hits"] = hits
        result["text_hit_count"] = len(hits)
    except Exception as e:
        result["error"] = repr(e)
    
    return result

# --------------------------------------------------------------------------------------
# Inventory files
# --------------------------------------------------------------------------------------

all_paths = []

for root in SEARCH_ROOTS:
    for path in root.rglob("*"):
        if path.is_file():
            ext = path.suffix.lower()
            if ext in DATA_EXTENSIONS or ext in TEXT_EXTENSIONS:
                all_paths.append(path)

print(f"Scanned roots: {[str(p) for p in SEARCH_ROOTS]}")
print(f"Files considered: {len(all_paths):,}")
print()

data_rows = []
text_rows = []

for path in all_paths:
    ext = path.suffix.lower()
    rel_path = str(path.relative_to(PROJECT_ROOT))
    pscore = path_score(path)
    size_mb = safe_file_size_mb(path)
    
    if ext in DATA_EXTENSIONS:
        metadata = {
            "rows": None,
            "cols": None,
            "columns": [],
            "error": None,
        }
        
        if ext == ".parquet":
            metadata = read_parquet_metadata(path)
        elif ext == ".csv":
            metadata = read_csv_metadata(path)
        elif ext == ".feather":
            metadata = read_feather_metadata(path)
        elif ext in {".pkl", ".pickle"}:
            metadata = read_pickle_metadata(path)
        
        columns = metadata.get("columns", []) or []
        detected_groups = detect_column_groups(columns)
        col_score = option_pricing_column_score(detected_groups)
        
        # Candidate if path suggests relevance OR columns suggest option pricing.
        is_candidate = (pscore >= 2) or (col_score >= 3)
        
        data_rows.append({
            "relative_path": rel_path,
            "full_path": str(path),
            "extension": ext,
            "size_mb": size_mb,
            "path_keyword_score": pscore,
            "option_pricing_column_score": col_score,
            "is_candidate": is_candidate,
            "rows": metadata.get("rows"),
            "cols": metadata.get("cols"),
            "date_cols": ", ".join(detected_groups.get("date", [])),
            "expiration_cols": ", ".join(detected_groups.get("expiration", [])),
            "strike_cols": ", ".join(detected_groups.get("strike", [])),
            "right_cols": ", ".join(detected_groups.get("right", [])),
            "bid_cols": ", ".join(detected_groups.get("bid", [])),
            "ask_cols": ", ".join(detected_groups.get("ask", [])),
            "mid_price_cols": ", ".join(detected_groups.get("mid", [])),
            "underlying_cols": ", ".join(detected_groups.get("underlying", [])),
            "columns": ", ".join([str(c) for c in columns]),
            "error": metadata.get("error"),
        })
    
    elif ext in TEXT_EXTENSIONS:
        # Only scan text files that have at least a weak path keyword score,
        # plus notebooks/scripts generally worth scanning.
        should_scan = (pscore >= 1) or (ext in {".py", ".ipynb"})
        
        if should_scan:
            text_scan = scan_text_file(path)
            is_candidate = pscore >= 1 or text_scan["text_hit_count"] >= 2
            
            text_rows.append({
                "relative_path": rel_path,
                "full_path": str(path),
                "extension": ext,
                "size_mb": size_mb,
                "path_keyword_score": pscore,
                "text_hit_count": text_scan["text_hit_count"],
                "text_hits": ", ".join(text_scan["text_hits"]),
                "is_candidate": is_candidate,
                "error": text_scan["error"],
            })

data_inventory = pd.DataFrame(data_rows)
text_inventory = pd.DataFrame(text_rows)

if data_inventory.empty:
    data_candidates = pd.DataFrame()
else:
    data_candidates = (
        data_inventory.loc[data_inventory["is_candidate"]]
        .sort_values(
            ["option_pricing_column_score", "path_keyword_score", "size_mb"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )

if text_inventory.empty:
    text_candidates = pd.DataFrame()
else:
    text_candidates = (
        text_inventory.loc[text_inventory["is_candidate"]]
        .sort_values(
            ["text_hit_count", "path_keyword_score", "size_mb"],
            ascending=[False, False, False],
        )
        .reset_index(drop=True)
    )

# --------------------------------------------------------------------------------------
# Try to identify strongest pricing-source candidates
# --------------------------------------------------------------------------------------

pricing_ready_mask = pd.Series(False, index=data_candidates.index)

if not data_candidates.empty:
    pricing_ready_mask = (
        data_candidates["date_cols"].astype(str).str.len().gt(0)
        & data_candidates["expiration_cols"].astype(str).str.len().gt(0)
        & data_candidates["strike_cols"].astype(str).str.len().gt(0)
        & data_candidates["bid_cols"].astype(str).str.len().gt(0)
        & data_candidates["ask_cols"].astype(str).str.len().gt(0)
    )

pricing_ready_candidates = data_candidates.loc[pricing_ready_mask].copy()

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

data_inventory_path = AUDIT_DIR / f"02B_option_price_source_data_inventory_all_{timestamp}.csv"
data_candidates_path = AUDIT_DIR / f"02B_option_price_source_data_candidates_{timestamp}.csv"
pricing_ready_path = AUDIT_DIR / f"02B_option_price_source_pricing_ready_candidates_{timestamp}.csv"
text_inventory_path = AUDIT_DIR / f"02B_option_price_source_text_inventory_all_{timestamp}.csv"
text_candidates_path = AUDIT_DIR / f"02B_option_price_source_text_candidates_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02B_option_price_source_inventory_manifest_{timestamp}.json"

data_inventory.to_csv(data_inventory_path, index=False)
data_candidates.to_csv(data_candidates_path, index=False)
pricing_ready_candidates.to_csv(pricing_ready_path, index=False)
text_inventory.to_csv(text_inventory_path, index=False)
text_candidates.to_csv(text_candidates_path, index=False)

manifest = {
    "cell": "2B",
    "description": "ThetaData / option price source inventory for SPY call spreads",
    "created_at": timestamp,
    "project_root": str(PROJECT_ROOT),
    "search_roots": [str(p) for p in SEARCH_ROOTS],
    "files_considered": len(all_paths),
    "data_files_inventory_count": int(len(data_inventory)),
    "data_candidate_count": int(len(data_candidates)),
    "pricing_ready_candidate_count": int(len(pricing_ready_candidates)),
    "text_files_inventory_count": int(len(text_inventory)),
    "text_candidate_count": int(len(text_candidates)),
    "data_inventory_path": str(data_inventory_path),
    "data_candidates_path": str(data_candidates_path),
    "pricing_ready_path": str(pricing_ready_path),
    "text_inventory_path": str(text_inventory_path),
    "text_candidates_path": str(text_candidates_path),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Inventory summary")
print("=" * 100)
print(f"Data files inventoried:              {len(data_inventory):,}")
print(f"Data candidates:                     {len(data_candidates):,}")
print(f"Pricing-ready candidates:            {len(pricing_ready_candidates):,}")
print(f"Text/script files inventoried:        {len(text_inventory):,}")
print(f"Text/script candidates:               {len(text_candidates):,}")
print()

print("Saved:")
print(f"  Data inventory all:                 {data_inventory_path}")
print(f"  Data candidates:                    {data_candidates_path}")
print(f"  Pricing-ready candidates:           {pricing_ready_path}")
print(f"  Text inventory all:                 {text_inventory_path}")
print(f"  Text/script candidates:             {text_candidates_path}")
print(f"  Manifest:                           {manifest_path}")

print()
print("=" * 100)
print("Pricing-ready data candidates")
print("=" * 100)

if pricing_ready_candidates.empty:
    print("No pricing-ready data files found with date + expiration + strike + bid + ask columns.")
else:
    display(
        pricing_ready_candidates.head(MAX_DISPLAY_ROWS)[
            [
                "relative_path",
                "extension",
                "rows",
                "cols",
                "size_mb",
                "option_pricing_column_score",
                "date_cols",
                "expiration_cols",
                "strike_cols",
                "right_cols",
                "bid_cols",
                "ask_cols",
                "mid_price_cols",
                "underlying_cols",
            ]
        ]
    )

print()
print("=" * 100)
print("Top data candidates")
print("=" * 100)

if data_candidates.empty:
    print("No data candidates found.")
else:
    display(
        data_candidates.head(MAX_DISPLAY_ROWS)[
            [
                "relative_path",
                "extension",
                "rows",
                "cols",
                "size_mb",
                "path_keyword_score",
                "option_pricing_column_score",
                "date_cols",
                "expiration_cols",
                "strike_cols",
                "right_cols",
                "bid_cols",
                "ask_cols",
                "mid_price_cols",
                "underlying_cols",
                "error",
            ]
        ]
    )

print()
print("=" * 100)
print("Top text/script candidates")
print("=" * 100)

if text_candidates.empty:
    print("No text/script candidates found.")
else:
    display(
        text_candidates.head(MAX_DISPLAY_ROWS)[
            [
                "relative_path",
                "extension",
                "size_mb",
                "path_keyword_score",
                "text_hit_count",
                "text_hits",
                "error",
            ]
        ]
    )

Cell 2B — ThetaData option price source inventory
Project root: C:\Users\patri\vrp_project
Audit dir:    C:\Users\patri\vrp_project\data\audit\call_sleeve_research

Scanned roots: ['C:\\Users\\patri\\vrp_project\\data', 'C:\\Users\\patri\\vrp_project\\notebooks']
Files considered: 14,515



In [ ]:
# Cell 2B.1 — Identify SPY option-chain availability
# Purpose:
#   Narrow the broad Cell 2B inventory to determine whether local SPY option-chain
#   files already exist and are usable for actual SPY call-spread entry pricing.
#
# Scope:
#   - Read-only inspection
#   - No ThetaData API pulls
#   - No option pricing yet
#   - No parameter sweep yet
#
# Goal:
#   Decide whether Cell 2C can price from existing local SPY chains,
#   or whether we first need a ThetaData SPY pull/cache step.

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Cell 2B.1 — Identify SPY option-chain availability")
print("=" * 100)
print(f"Project root: {PROJECT_ROOT}")
print(f"Audit dir:    {AUDIT_DIR}")
print()

# --------------------------------------------------------------------------------------
# Locate latest Cell 2B pricing-ready inventory
# --------------------------------------------------------------------------------------

pricing_ready_files = sorted(
    AUDIT_DIR.glob("02B_option_price_source_pricing_ready_candidates_*.csv"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not pricing_ready_files:
    raise FileNotFoundError(
        "No Cell 2B pricing-ready inventory found. Run Cell 2B first."
    )

PRICING_READY_PATH = pricing_ready_files[0]

print(f"Using pricing-ready inventory: {PRICING_READY_PATH}")
print()

inv = pd.read_csv(PRICING_READY_PATH)

if inv.empty:
    raise ValueError("Pricing-ready inventory is empty.")

required_inv_cols = ["relative_path", "full_path", "extension", "rows", "cols"]
missing_inv_cols = [c for c in required_inv_cols if c not in inv.columns]
if missing_inv_cols:
    raise ValueError(f"Pricing-ready inventory missing required columns: {missing_inv_cols}")

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def normalize_path_string(x):
    return str(x).replace("/", "\\").lower()

def infer_root_from_filename(path_str: str):
    """
    Infer root from filename patterns like:
      SPY_20260529_20260619...
      SPX_20260529_20260619...
      SPXW_20260529_20260619...
    """
    name = Path(str(path_str)).name.upper()
    
    if re.match(r"^SPY[_\-]", name):
        return "SPY"
    if re.match(r"^SPXW[_\-]", name):
        return "SPXW"
    if re.match(r"^SPX[_\-]", name):
        return "SPX"
    
    # Slightly looser fallback, but avoid classifying SPX/SPXW as SPY.
    if "SPY" in name and "SPX" not in name:
        return "SPY"
    if "SPXW" in name:
        return "SPXW"
    if "SPX" in name:
        return "SPX"
    
    return "UNKNOWN"

def parse_dates_from_filename(path_str: str):
    """
    Try to pull all YYYYMMDD tokens from filename.
    Usually first token is quote/trade date and second token may be expiration.
    """
    name = Path(str(path_str)).name
    tokens = re.findall(r"(20\d{6})", name)
    dates = []
    for t in tokens:
        try:
            dates.append(pd.to_datetime(t, format="%Y%m%d"))
        except Exception:
            pass
    
    return dates

def load_option_file(path: Path):
    ext = path.suffix.lower()
    if ext == ".pkl" or ext == ".pickle":
        return pd.read_pickle(path)
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext == ".csv":
        return pd.read_csv(path)
    if ext == ".feather":
        return pd.read_feather(path)
    raise ValueError(f"Unsupported extension for sample load: {ext}")

def safe_unique_values(df, col, max_vals=20):
    if col not in df.columns:
        return []
    vals = pd.Series(df[col]).dropna().unique().tolist()
    vals = sorted(vals, key=lambda x: str(x))
    return vals[:max_vals]

def safe_min_max_date(df, col):
    if col not in df.columns:
        return (None, None)
    s = pd.to_datetime(df[col], errors="coerce")
    if s.notna().sum() == 0:
        return (None, None)
    return (s.min(), s.max())

def safe_min_max_numeric(df, col):
    if col not in df.columns:
        return (None, None)
    s = pd.to_numeric(df[col], errors="coerce")
    if s.notna().sum() == 0:
        return (None, None)
    return (s.min(), s.max())

def find_first_existing_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# --------------------------------------------------------------------------------------
# Classify inventory by root
# --------------------------------------------------------------------------------------

inv["root_from_filename"] = inv["relative_path"].apply(infer_root_from_filename)

inv["filename_dates"] = inv["relative_path"].apply(parse_dates_from_filename)
inv["filename_first_date"] = inv["filename_dates"].apply(lambda xs: xs[0] if len(xs) >= 1 else pd.NaT)
inv["filename_second_date"] = inv["filename_dates"].apply(lambda xs: xs[1] if len(xs) >= 2 else pd.NaT)

inv["file_exists"] = inv["full_path"].apply(lambda x: Path(str(x)).exists())

root_summary = (
    inv
    .groupby(["root_from_filename", "extension"], as_index=False)
    .agg(
        file_count=("relative_path", "count"),
        existing_file_count=("file_exists", "sum"),
        total_rows=("rows", "sum"),
        min_filename_first_date=("filename_first_date", "min"),
        max_filename_first_date=("filename_first_date", "max"),
        min_filename_second_date=("filename_second_date", "min"),
        max_filename_second_date=("filename_second_date", "max"),
    )
    .sort_values(["root_from_filename", "extension"])
)

spy_candidates = (
    inv.loc[inv["root_from_filename"] == "SPY"]
    .sort_values(["filename_first_date", "filename_second_date", "relative_path"])
    .reset_index(drop=True)
)

spx_candidates = inv.loc[inv["root_from_filename"].isin(["SPX", "SPXW"])].copy()
unknown_candidates = inv.loc[inv["root_from_filename"] == "UNKNOWN"].copy()

# --------------------------------------------------------------------------------------
# SPY date coverage from filenames
# --------------------------------------------------------------------------------------

if not spy_candidates.empty:
    spy_date_coverage = (
        spy_candidates
        .groupby("filename_first_date", as_index=False)
        .agg(
            file_count=("relative_path", "count"),
            min_expiry_hint=("filename_second_date", "min"),
            max_expiry_hint=("filename_second_date", "max"),
            total_rows=("rows", "sum"),
        )
        .sort_values("filename_first_date")
    )
else:
    spy_date_coverage = pd.DataFrame(
        columns=[
            "filename_first_date",
            "file_count",
            "min_expiry_hint",
            "max_expiry_hint",
            "total_rows",
        ]
    )

# --------------------------------------------------------------------------------------
# Sample-load SPY files if found
# --------------------------------------------------------------------------------------

sample_rows = []
sampled_dfs_preview = []

if not spy_candidates.empty:
    # Sample first 3, middle 3, last 3 unique files to check actual root/right/bid/ask structure.
    n = len(spy_candidates)
    sample_indices = sorted(set(
        list(range(min(3, n)))
        + [max(0, n // 2 - 1), n // 2, min(n - 1, n // 2 + 1)]
        + list(range(max(0, n - 3), n))
    ))
    
    sample_candidates = spy_candidates.iloc[sample_indices].copy()
    
    for _, row in sample_candidates.iterrows():
        path = Path(row["full_path"])
        result = {
            "relative_path": row["relative_path"],
            "full_path": str(path),
            "extension": path.suffix.lower(),
            "file_exists": path.exists(),
            "load_ok": False,
            "rows": None,
            "cols": None,
            "columns": None,
            "root_values": None,
            "right_values": None,
            "expiration_min": None,
            "expiration_max": None,
            "timestamp_min": None,
            "timestamp_max": None,
            "strike_min": None,
            "strike_max": None,
            "bid_min": None,
            "bid_max": None,
            "ask_min": None,
            "ask_max": None,
            "mid_min": None,
            "mid_max": None,
            "bid_missing": None,
            "ask_missing": None,
            "error": None,
        }
        
        try:
            df_sample = load_option_file(path)
            
            result["load_ok"] = True
            result["rows"] = len(df_sample)
            result["cols"] = len(df_sample.columns)
            result["columns"] = ", ".join(map(str, df_sample.columns))
            
            root_col = find_first_existing_col(df_sample, ["root", "underlying", "symbol", "ticker"])
            right_col = find_first_existing_col(df_sample, ["right", "option_type", "cp", "call_put", "put_call"])
            exp_col = find_first_existing_col(df_sample, ["expiration", "expiry", "expiration_date", "exp"])
            ts_col = find_first_existing_col(df_sample, ["timestamp", "quote_date", "trade_date", "date", "datetime"])
            strike_col = find_first_existing_col(df_sample, ["strike"])
            bid_col = find_first_existing_col(df_sample, ["bid", "bid_price"])
            ask_col = find_first_existing_col(df_sample, ["ask", "ask_price"])
            mid_col = find_first_existing_col(df_sample, ["mid", "mark"])
            
            result["root_values"] = ", ".join(map(str, safe_unique_values(df_sample, root_col))) if root_col else None
            result["right_values"] = ", ".join(map(str, safe_unique_values(df_sample, right_col))) if right_col else None
            
            exp_min, exp_max = safe_min_max_date(df_sample, exp_col) if exp_col else (None, None)
            ts_min, ts_max = safe_min_max_date(df_sample, ts_col) if ts_col else (None, None)
            strike_min, strike_max = safe_min_max_numeric(df_sample, strike_col) if strike_col else (None, None)
            bid_min, bid_max = safe_min_max_numeric(df_sample, bid_col) if bid_col else (None, None)
            ask_min, ask_max = safe_min_max_numeric(df_sample, ask_col) if ask_col else (None, None)
            mid_min, mid_max = safe_min_max_numeric(df_sample, mid_col) if mid_col else (None, None)
            
            result["expiration_min"] = exp_min
            result["expiration_max"] = exp_max
            result["timestamp_min"] = ts_min
            result["timestamp_max"] = ts_max
            result["strike_min"] = strike_min
            result["strike_max"] = strike_max
            result["bid_min"] = bid_min
            result["bid_max"] = bid_max
            result["ask_min"] = ask_min
            result["ask_max"] = ask_max
            result["mid_min"] = mid_min
            result["mid_max"] = mid_max
            
            result["bid_missing"] = int(df_sample[bid_col].isna().sum()) if bid_col else None
            result["ask_missing"] = int(df_sample[ask_col].isna().sum()) if ask_col else None
            
            # Save a tiny preview with source path attached.
            preview = df_sample.head(5).copy()
            preview.insert(0, "source_relative_path", row["relative_path"])
            sampled_dfs_preview.append(preview)
            
        except Exception as e:
            result["error"] = repr(e)
        
        sample_rows.append(result)

spy_sample_summary = pd.DataFrame(sample_rows)

if sampled_dfs_preview:
    spy_sample_preview = pd.concat(sampled_dfs_preview, ignore_index=True)
else:
    spy_sample_preview = pd.DataFrame()

# --------------------------------------------------------------------------------------
# Save outputs
# --------------------------------------------------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

classified_inventory_path = AUDIT_DIR / f"02B1_option_chain_inventory_classified_{timestamp}.csv"
root_summary_path = AUDIT_DIR / f"02B1_option_chain_root_summary_{timestamp}.csv"
spy_candidates_path = AUDIT_DIR / f"02B1_spy_option_chain_candidates_{timestamp}.csv"
spy_date_coverage_path = AUDIT_DIR / f"02B1_spy_option_chain_date_coverage_{timestamp}.csv"
spy_sample_summary_path = AUDIT_DIR / f"02B1_spy_option_chain_sample_summary_{timestamp}.csv"
spy_sample_preview_path = AUDIT_DIR / f"02B1_spy_option_chain_sample_preview_{timestamp}.csv"
manifest_path = AUDIT_DIR / f"02B1_spy_option_chain_availability_manifest_{timestamp}.json"

inv.to_csv(classified_inventory_path, index=False)
root_summary.to_csv(root_summary_path, index=False)
spy_candidates.to_csv(spy_candidates_path, index=False)
spy_date_coverage.to_csv(spy_date_coverage_path, index=False)
spy_sample_summary.to_csv(spy_sample_summary_path, index=False)
spy_sample_preview.to_csv(spy_sample_preview_path, index=False)

manifest = {
    "cell": "2B.1",
    "description": "Identify local SPY option-chain availability",
    "created_at": timestamp,
    "pricing_ready_inventory_path": str(PRICING_READY_PATH),
    "classified_inventory_path": str(classified_inventory_path),
    "root_summary_path": str(root_summary_path),
    "spy_candidates_path": str(spy_candidates_path),
    "spy_date_coverage_path": str(spy_date_coverage_path),
    "spy_sample_summary_path": str(spy_sample_summary_path),
    "spy_sample_preview_path": str(spy_sample_preview_path),
    "pricing_ready_rows": int(len(inv)),
    "spy_candidate_files": int(len(spy_candidates)),
    "spx_spxw_candidate_files": int(len(spx_candidates)),
    "unknown_candidate_files": int(len(unknown_candidates)),
    "spy_unique_quote_dates_from_filename": int(spy_candidates["filename_first_date"].nunique()) if not spy_candidates.empty else 0,
    "spy_filename_date_min": str(spy_candidates["filename_first_date"].min().date()) if not spy_candidates.empty and pd.notna(spy_candidates["filename_first_date"].min()) else None,
    "spy_filename_date_max": str(spy_candidates["filename_first_date"].max().date()) if not spy_candidates.empty and pd.notna(spy_candidates["filename_first_date"].max()) else None,
    "recommendation": (
        "SPY option chains found locally; inspect sample summary and proceed to Cell 2C pricing cache."
        if not spy_candidates.empty
        else "No SPY option chains found locally; build a ThetaData SPY pull/cache step before Cell 2C."
    ),
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

# --------------------------------------------------------------------------------------
# Console output
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Root summary")
print("=" * 100)
display(root_summary)

print()
print("=" * 100)
print("SPY availability summary")
print("=" * 100)

print(f"SPY candidate files:                 {len(spy_candidates):,}")
print(f"SPX/SPXW candidate files:            {len(spx_candidates):,}")
print(f"Unknown candidate files:             {len(unknown_candidates):,}")

if spy_candidates.empty:
    print()
    print("Result: No local SPY option-chain files found in the pricing-ready inventory.")
    print("Next step: build a ThetaData SPY option-chain pull/cache cell before pricing.")
else:
    print(f"SPY filename quote-date min:          {spy_candidates['filename_first_date'].min().date()}")
    print(f"SPY filename quote-date max:          {spy_candidates['filename_first_date'].max().date()}")
    print(f"SPY unique quote dates from filename: {spy_candidates['filename_first_date'].nunique():,}")
    print()
    print("Result: Local SPY option-chain candidates exist.")
    print("Next step: inspect sample summary, then build Cell 2C priced candidate cache.")

print()
print("Saved:")
print(f"  Classified inventory:              {classified_inventory_path}")
print(f"  Root summary:                      {root_summary_path}")
print(f"  SPY candidates:                    {spy_candidates_path}")
print(f"  SPY date coverage:                 {spy_date_coverage_path}")
print(f"  SPY sample summary:                {spy_sample_summary_path}")
print(f"  SPY sample preview:                {spy_sample_preview_path}")
print(f"  Manifest:                          {manifest_path}")

print()
print("=" * 100)
print("SPY date coverage preview")
print("=" * 100)

if spy_date_coverage.empty:
    print("No SPY date coverage to display.")
else:
    display(spy_date_coverage.head(20))
    print("...")
    display(spy_date_coverage.tail(20))

print()
print("=" * 100)
print("SPY candidate files preview")
print("=" * 100)

if spy_candidates.empty:
    print("No SPY candidates.")
else:
    display(
        pd.concat(
            [
                spy_candidates.head(20),
                spy_candidates.tail(20),
            ],
            ignore_index=True,
        )
        .drop_duplicates(subset=["relative_path"])
        [
            [
                "relative_path",
                "extension",
                "rows",
                "cols",
                "size_mb",
                "filename_first_date",
                "filename_second_date",
                "date_cols",
                "expiration_cols",
                "strike_cols",
                "right_cols",
                "bid_cols",
                "ask_cols",
                "mid_price_cols",
                "underlying_cols",
            ]
        ]
    )

print()
print("=" * 100)
print("SPY sample file summary")
print("=" * 100)

if spy_sample_summary.empty:
    print("No SPY sample files loaded.")
else:
    display(spy_sample_summary)

print()
print("=" * 100)
print("SPY sample preview rows")
print("=" * 100)

if spy_sample_preview.empty:
    print("No SPY sample preview rows.")
else:
    display(spy_sample_preview.head(30))

In [ ]:
# Cell 2B.2 — Inspect ThetaData pull code and test one SPY chain pull
# Purpose:
#   We confirmed no local SPY option chains exist. This cell:
#   1. Inspects existing ThetaData-related scripts/notebooks for endpoint patterns.
#   2. Inspects an existing SPX/SPXW chain file to confirm local saved schema.
#   3. Attempts one small SPY chain pull from local ThetaData.
#   4. Saves the test SPY chain only if the result has usable quote columns.
#
# Scope:
#   - Tiny probe only
#   - No historical bulk SPY download
#   - No parameter sweep
#   - No priced candidate cache yet
#
# Output folder:
#   data\raw\thetadata_chains_spy
#
# Expected saved naming:
#   SPY_YYYYMMDD_YYYYMMDD.pkl
#
# Notes:
#   - This is intentionally defensive because we need to infer the exact local ThetaData endpoint format.
#   - If the pull fails, paste the console output and we will adapt the endpoint.

from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
import requests
from io import StringIO
from datetime import datetime, timedelta

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")

AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
SPY_CHAIN_DIR = PROJECT_ROOT / "data" / "raw" / "thetadata_chains_spy"
EXISTING_CHAIN_DIR = PROJECT_ROOT / "data" / "raw" / "thetadata_chains"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SPY_CHAIN_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Cell 2B.2 — Inspect ThetaData pull code and test one SPY chain pull")
print("=" * 100)
print(f"Project root:            {PROJECT_ROOT}")
print(f"SPY chain output dir:    {SPY_CHAIN_DIR}")
print()

# --------------------------------------------------------------------------------------
# Config
# --------------------------------------------------------------------------------------

THETADATA_BASE_URLS = [
    "http://127.0.0.1:25510",
    "http://localhost:25510",
]

# Pick a recent date that should exist in your local final panel.
# This is only a one-chain probe. It does not need to be a selected trade.
RAW_CANDIDATE_PANEL_PATH = (
    PROCESSED_DIR / "call_sleeve_corsi_all_tenor_raw_candidate_panel_v1.parquet"
)

TEST_ROOT = "SPY"
TEST_TENOR = 30

# If None, auto-pick latest complete candidate date from Cell 2A.
OVERRIDE_TEST_TRADE_DATE = None  # Example: "2026-07-01"

# Endpoint candidates. We try these in order.
# Some ThetaData installs support /v2/list/expirations and /v2/bulk_hist/option/quote.
EXPIRATION_ENDPOINT_CANDIDATES = [
    "/v2/list/expirations",
    "/v2/list/expirations/option",
]

CHAIN_ENDPOINT_CANDIDATES = [
    "/v2/bulk_hist/option/quote",
    "/v2/bulk_hist/option/eod",
]

# Quote interval guesses. 900000 ms = 15 minutes. Existing files may have one snapshot.
# If an endpoint ignores this, harmless.
IVL_CANDIDATES = [900000, 0]

# --------------------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------------------

def yyyymmdd(dt) -> str:
    return pd.Timestamp(dt).strftime("%Y%m%d")

def normalize_expiration_value(x):
    """
    Convert likely expiration representations into YYYYMMDD string if possible.
    """
    if pd.isna(x):
        return None
    
    s = str(x).strip()
    
    # Already YYYYMMDD
    if re.fullmatch(r"20\d{6}", s):
        return s
    
    # Numeric like 20260731.0
    if re.fullmatch(r"20\d{6}\.0", s):
        return s.split(".")[0]
    
    # ISO date
    try:
        return pd.to_datetime(s).strftime("%Y%m%d")
    except Exception:
        return None

def response_to_dataframe(resp: requests.Response) -> pd.DataFrame:
    """
    Parse common ThetaData response formats:
    - JSON with header/response
    - JSON list/dict
    - CSV text
    """
    text = resp.text.strip()
    
    if not text:
        return pd.DataFrame()
    
    # Try JSON first
    try:
        obj = resp.json()
        
        if isinstance(obj, dict):
            # Common pattern: {"header": {"format": [...]}, "response": [[...], ...]}
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None
                
                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break
                
                if cols is not None and isinstance(data, list):
                    return pd.DataFrame(data, columns=cols)
                return pd.DataFrame(data)
            
            # Otherwise flatten dict.
            return pd.json_normalize(obj)
        
        if isinstance(obj, list):
            return pd.DataFrame(obj)
    
    except Exception:
        pass
    
    # Try CSV
    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        pass
    
    # Last resort: line split
    return pd.DataFrame({"raw_response": text.splitlines()})

def request_df(base_url, endpoint, params, timeout=30):
    url = base_url.rstrip("/") + endpoint
    resp = requests.get(url, params=params, timeout=timeout)
    status = resp.status_code
    df = pd.DataFrame()
    error = None
    
    try:
        resp.raise_for_status()
        df = response_to_dataframe(resp)
    except Exception as e:
        error = repr(e)
    
    return {
        "url": url,
        "params": params,
        "status_code": status,
        "ok": error is None,
        "rows": len(df) if isinstance(df, pd.DataFrame) else 0,
        "cols": len(df.columns) if isinstance(df, pd.DataFrame) else 0,
        "df": df,
        "error": error,
        "text_preview": resp.text[:500] if hasattr(resp, "text") else "",
    }

def first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    
    # Case-insensitive fallback
    lower_map = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    
    return None

def has_usable_chain_schema(df):
    if df is None or df.empty:
        return False
    
    strike_col = first_col(df, ["strike"])
    right_col = first_col(df, ["right", "option_type", "cp", "call_put", "put_call"])
    bid_col = first_col(df, ["bid", "bid_price"])
    ask_col = first_col(df, ["ask", "ask_price"])
    
    return all(c is not None for c in [strike_col, right_col, bid_col, ask_col])

def standardize_chain_schema(df, root, quote_date, expiration):
    out = df.copy()
    
    col_map = {}
    for canonical, candidates in {
        "timestamp": ["timestamp", "datetime", "date", "trade_date", "quote_date"],
        "root": ["root", "underlying", "symbol", "ticker"],
        "expiration": ["expiration", "expiry", "expiration_date", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type", "cp", "call_put", "put_call"],
        "bid": ["bid", "bid_price"],
        "ask": ["ask", "ask_price"],
        "mid": ["mid", "mark"],
    }.items():
        c = first_col(out, candidates)
        if c is not None:
            col_map[c] = canonical
    
    out = out.rename(columns=col_map)
    
    if "root" not in out.columns:
        out["root"] = root
    
    if "expiration" not in out.columns:
        out["expiration"] = expiration
    
    if "timestamp" not in out.columns:
        out["timestamp"] = quote_date
    
    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (pd.to_numeric(out["bid"], errors="coerce") + pd.to_numeric(out["ask"], errors="coerce")) / 2.0
    
    # Normalize key fields
    out["root"] = out["root"].astype(str).str.upper()
    out["expiration"] = out["expiration"].apply(normalize_expiration_value)
    out["strike"] = pd.to_numeric(out["strike"], errors="coerce")
    out["bid"] = pd.to_numeric(out["bid"], errors="coerce")
    out["ask"] = pd.to_numeric(out["ask"], errors="coerce")
    out["mid"] = pd.to_numeric(out["mid"], errors="coerce")
    
    if "right" in out.columns:
        out["right"] = out["right"].astype(str).str.upper().str[0]
    
    return out

def find_candidate_expirations_from_response(df, target_date):
    """
    Extract expiration candidates from an expiration-list response.
    """
    if df is None or df.empty:
        return []
    
    candidates = []
    
    for col in df.columns:
        vals = df[col].dropna().tolist()
        for v in vals:
            exp = normalize_expiration_value(v)
            if exp is not None:
                candidates.append(exp)
    
    candidates = sorted(set(candidates))
    
    # Keep expirations near the target date.
    target = pd.Timestamp(target_date)
    near = []
    for exp in candidates:
        dt = pd.to_datetime(exp, format="%Y%m%d", errors="coerce")
        if pd.notna(dt) and abs((dt - target).days) <= 14:
            near.append(exp)
    
    return near

def fallback_expirations_around_target(target_date, radius_days=10):
    """
    Generate likely SPY expirations around target date.
    SPY has frequent expiries; this fallback tries every business day around target.
    """
    target = pd.Timestamp(target_date)
    dates = pd.date_range(target - pd.Timedelta(days=radius_days), target + pd.Timedelta(days=radius_days), freq="B")
    return [yyyymmdd(d) for d in dates]

# --------------------------------------------------------------------------------------
# 1) Inspect existing ThetaData-related scripts/notebooks
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 1 — Existing ThetaData code snippets")
print("=" * 100)

text_paths = []
for root in [PROJECT_ROOT / "notebooks", PROJECT_ROOT / "scripts"]:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".py", ".ipynb", ".txt", ".md"}:
                text_paths.append(p)

terms = [
    "bulk_hist",
    "hist/option",
    "list/expirations",
    "thetadata",
    "127.0.0.1",
    "25510",
    "bid",
    "ask",
]

snippet_rows = []

for path in text_paths:
    try:
        text = path.read_text(errors="ignore")
    except Exception:
        continue
    
    low = text.lower()
    if not any(t.lower() in low for t in terms):
        continue
    
    lines = text.splitlines()
    for i, line in enumerate(lines):
        if any(t.lower() in line.lower() for t in terms):
            start = max(0, i - 2)
            end = min(len(lines), i + 3)
            snippet = "\n".join(lines[start:end])
            snippet_rows.append({
                "relative_path": str(path.relative_to(PROJECT_ROOT)),
                "line_number": i + 1,
                "matched_line": line.strip()[:300],
                "snippet": snippet[:1500],
            })

snippet_df = pd.DataFrame(snippet_rows)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
snippet_path = AUDIT_DIR / f"02B2_thetadata_code_snippets_{timestamp}.csv"
snippet_df.to_csv(snippet_path, index=False)

print(f"Snippet rows found: {len(snippet_df):,}")
print(f"Saved snippets:     {snippet_path}")

if not snippet_df.empty:
    display(snippet_df.head(30)[["relative_path", "line_number", "matched_line"]])
else:
    print("No ThetaData snippets found.")

print()

# --------------------------------------------------------------------------------------
# 2) Inspect existing local SPX/SPXW chain schema
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 2 — Existing SPX/SPXW chain schema sample")
print("=" * 100)

existing_chain_files = []
if EXISTING_CHAIN_DIR.exists():
    existing_chain_files = sorted(
        list(EXISTING_CHAIN_DIR.glob("SPX_*.pkl")) + list(EXISTING_CHAIN_DIR.glob("SPXW_*.pkl")),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

existing_schema_summary = pd.DataFrame()
existing_preview = pd.DataFrame()

if existing_chain_files:
    sample_file = existing_chain_files[0]
    try:
        sample_df = pd.read_pickle(sample_file)
        
        existing_schema_summary = pd.DataFrame([{
            "sample_file": str(sample_file.relative_to(PROJECT_ROOT)),
            "rows": len(sample_df),
            "cols": len(sample_df.columns),
            "columns": ", ".join(map(str, sample_df.columns)),
            "root_values": ", ".join(map(str, sorted(sample_df["root"].dropna().unique().tolist()))) if "root" in sample_df.columns else None,
            "right_values": ", ".join(map(str, sorted(sample_df["right"].dropna().unique().tolist()))) if "right" in sample_df.columns else None,
            "expiration_values": ", ".join(map(str, sorted(sample_df["expiration"].dropna().astype(str).unique().tolist())[:10])) if "expiration" in sample_df.columns else None,
            "timestamp_values": ", ".join(map(str, sorted(sample_df["timestamp"].dropna().astype(str).unique().tolist())[:10])) if "timestamp" in sample_df.columns else None,
            "strike_min": pd.to_numeric(sample_df["strike"], errors="coerce").min() if "strike" in sample_df.columns else np.nan,
            "strike_max": pd.to_numeric(sample_df["strike"], errors="coerce").max() if "strike" in sample_df.columns else np.nan,
            "bid_missing": int(sample_df["bid"].isna().sum()) if "bid" in sample_df.columns else None,
            "ask_missing": int(sample_df["ask"].isna().sum()) if "ask" in sample_df.columns else None,
        }])
        
        existing_preview = sample_df.head(20).copy()
        
        display(existing_schema_summary)
        display(existing_preview)
    
    except Exception as e:
        print(f"Failed to inspect sample existing chain file {sample_file}: {repr(e)}")
else:
    print(f"No existing SPX/SPXW chain files found under {EXISTING_CHAIN_DIR}")

existing_schema_path = AUDIT_DIR / f"02B2_existing_chain_schema_summary_{timestamp}.csv"
existing_preview_path = AUDIT_DIR / f"02B2_existing_chain_preview_{timestamp}.csv"
existing_schema_summary.to_csv(existing_schema_path, index=False)
existing_preview.to_csv(existing_preview_path, index=False)

print(f"Saved existing schema summary: {existing_schema_path}")
print(f"Saved existing preview:        {existing_preview_path}")
print()

# --------------------------------------------------------------------------------------
# 3) Choose one test trade date / target expiration
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 3 — Choose one SPY chain test date")
print("=" * 100)

if OVERRIDE_TEST_TRADE_DATE is not None:
    test_trade_date = pd.Timestamp(OVERRIDE_TEST_TRADE_DATE)
else:
    raw_panel = pd.read_parquet(RAW_CANDIDATE_PANEL_PATH)
    raw_panel["trade_date"] = pd.to_datetime(raw_panel["trade_date"], errors="coerce")
    
    complete = raw_panel.loc[
        (raw_panel["tenor"] == TEST_TENOR)
        & (raw_panel["candidate_inputs_complete"])
    ].copy()
    
    if complete.empty:
        raise ValueError("No complete 30D candidates found to choose a test trade date.")
    
    test_trade_date = complete["trade_date"].max()

target_expiration_date = test_trade_date + pd.Timedelta(days=TEST_TENOR)

print(f"Test root:              {TEST_ROOT}")
print(f"Test trade date:        {test_trade_date.date()}")
print(f"Target expiration date: {target_expiration_date.date()}")
print()

# --------------------------------------------------------------------------------------
# 4) Check ThetaData local service availability
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 4 — Probe local ThetaData service")
print("=" * 100)

service_probe_rows = []
working_base_url = None

for base in THETADATA_BASE_URLS:
    for endpoint in ["/v2/system/status", "/v2/list/roots/option", "/"]:
        try:
            url = base.rstrip("/") + endpoint
            resp = requests.get(url, timeout=5)
            service_probe_rows.append({
                "base_url": base,
                "endpoint": endpoint,
                "url": url,
                "status_code": resp.status_code,
                "ok": resp.ok,
                "text_preview": resp.text[:300],
                "error": None,
            })
            
            if resp.status_code < 500 and working_base_url is None:
                working_base_url = base
        
        except Exception as e:
            service_probe_rows.append({
                "base_url": base,
                "endpoint": endpoint,
                "url": base.rstrip("/") + endpoint,
                "status_code": None,
                "ok": False,
                "text_preview": "",
                "error": repr(e),
            })

service_probe_df = pd.DataFrame(service_probe_rows)
service_probe_path = AUDIT_DIR / f"02B2_thetadata_service_probe_{timestamp}.csv"
service_probe_df.to_csv(service_probe_path, index=False)

display(service_probe_df)

if working_base_url is None:
    print()
    print("No local ThetaData service endpoint responded.")
    print("Start ThetaData Terminal / local API, then rerun this cell.")
else:
    print(f"Working base URL candidate: {working_base_url}")

print()

# --------------------------------------------------------------------------------------
# 5) Pull expiration list if possible
# --------------------------------------------------------------------------------------

expiration_probe_rows = []
expiration_candidates = []

if working_base_url is not None:
    print("=" * 100)
    print("Step 5 — Try expiration-list endpoints")
    print("=" * 100)
    
    for endpoint in EXPIRATION_ENDPOINT_CANDIDATES:
        params = {"root": TEST_ROOT}
        result = request_df(working_base_url, endpoint, params, timeout=20)
        df_exp = result["df"]
        
        near_exp = find_candidate_expirations_from_response(df_exp, target_expiration_date)
        
        expiration_probe_rows.append({
            "endpoint": endpoint,
            "params": json.dumps(params),
            "status_code": result["status_code"],
            "ok": result["ok"],
            "rows": result["rows"],
            "cols": result["cols"],
            "near_expiration_count": len(near_exp),
            "near_expirations": ", ".join(near_exp[:30]),
            "columns": ", ".join(map(str, df_exp.columns)) if isinstance(df_exp, pd.DataFrame) else "",
            "error": result["error"],
            "text_preview": result["text_preview"],
        })
        
        expiration_candidates.extend(near_exp)
    
    expiration_candidates = sorted(set(expiration_candidates))
    
    if not expiration_candidates:
        expiration_candidates = fallback_expirations_around_target(target_expiration_date, radius_days=10)
        expiration_source = "fallback_business_days_around_target"
    else:
        expiration_source = "thetadata_expiration_list"
    
    expiration_probe_df = pd.DataFrame(expiration_probe_rows)
    display(expiration_probe_df)
    
    print()
    print(f"Expiration source:       {expiration_source}")
    print(f"Expiration candidates:   {expiration_candidates[:30]}")
else:
    expiration_probe_df = pd.DataFrame()
    expiration_source = "no_service"
    expiration_candidates = []

expiration_probe_path = AUDIT_DIR / f"02B2_spy_expiration_probe_{timestamp}.csv"
expiration_probe_df.to_csv(expiration_probe_path, index=False)
print()

# --------------------------------------------------------------------------------------
# 6) Try one SPY chain pull
# --------------------------------------------------------------------------------------

print("=" * 100)
print("Step 6 — Try one SPY chain pull")
print("=" * 100)

chain_attempt_rows = []
saved_chain_path = None
saved_chain_df = pd.DataFrame()

if working_base_url is not None and expiration_candidates:
    test_trade_yyyymmdd = yyyymmdd(test_trade_date)
    
    # Closest expirations first
    expiration_candidates_sorted = sorted(
        expiration_candidates,
        key=lambda exp: abs((pd.to_datetime(exp, format="%Y%m%d") - target_expiration_date).days)
    )
    
    for exp in expiration_candidates_sorted[:20]:
        if saved_chain_path is not None:
            break
        
        for endpoint in CHAIN_ENDPOINT_CANDIDATES:
            if saved_chain_path is not None:
                break
            
            for ivl in IVL_CANDIDATES:
                params = {
                    "root": TEST_ROOT,
                    "exp": exp,
                    "start_date": test_trade_yyyymmdd,
                    "end_date": test_trade_yyyymmdd,
                }
                
                # Try with ivl; if endpoint ignores it, fine.
                if ivl is not None:
                    params["ivl"] = ivl
                
                result = request_df(working_base_url, endpoint, params, timeout=60)
                df_chain = result["df"]
                
                usable = has_usable_chain_schema(df_chain)
                
                chain_attempt_rows.append({
                    "endpoint": endpoint,
                    "exp": exp,
                    "ivl": ivl,
                    "params": json.dumps(params),
                    "status_code": result["status_code"],
                    "ok": result["ok"],
                    "rows": result["rows"],
                    "cols": result["cols"],
                    "usable_schema": usable,
                    "columns": ", ".join(map(str, df_chain.columns)) if isinstance(df_chain, pd.DataFrame) else "",
                    "error": result["error"],
                    "text_preview": result["text_preview"],
                })
                
                if usable and not df_chain.empty:
                    standardized = standardize_chain_schema(
                        df_chain,
                        root=TEST_ROOT,
                        quote_date=test_trade_yyyymmdd,
                        expiration=exp,
                    )
                    
                    # Keep rows with valid strike/right/bid/ask.
                    valid = standardized[
                        standardized["strike"].notna()
                        & standardized["right"].notna()
                        & standardized["bid"].notna()
                        & standardized["ask"].notna()
                    ].copy()
                    
                    if not valid.empty:
                        # Save exactly what we got, standardized.
                        out_name = f"{TEST_ROOT}_{test_trade_yyyymmdd}_{exp}.pkl"
                        saved_chain_path = SPY_CHAIN_DIR / out_name
                        valid.to_pickle(saved_chain_path)
                        saved_chain_df = valid
                        break

chain_attempt_df = pd.DataFrame(chain_attempt_rows)
chain_attempt_path = AUDIT_DIR / f"02B2_spy_chain_pull_attempts_{timestamp}.csv"
chain_attempt_df.to_csv(chain_attempt_path, index=False)

if chain_attempt_df.empty:
    print("No chain pull attempts were made.")
else:
    display(chain_attempt_df[[
        "endpoint",
        "exp",
        "ivl",
        "status_code",
        "ok",
        "rows",
        "cols",
        "usable_schema",
        "columns",
        "error",
    ]].head(100))

print()

if saved_chain_path is None:
    print("No usable SPY chain was saved.")
    print("Paste the chain attempt table and text previews if needed; we will adapt the endpoint.")
else:
    print(f"Saved usable SPY test chain: {saved_chain_path}")
    print(f"Saved rows:                  {len(saved_chain_df):,}")
    print(f"Columns:                     {list(saved_chain_df.columns)}")
    print()
    print("Saved chain summary:")
    
    saved_summary = pd.DataFrame([{
        "path": str(saved_chain_path),
        "rows": len(saved_chain_df),
        "root_values": ", ".join(map(str, sorted(saved_chain_df["root"].dropna().unique().tolist()))),
        "right_values": ", ".join(map(str, sorted(saved_chain_df["right"].dropna().unique().tolist()))),
        "expiration_values": ", ".join(map(str, sorted(saved_chain_df["expiration"].dropna().unique().tolist()))),
        "timestamp_values_preview": ", ".join(map(str, sorted(saved_chain_df["timestamp"].dropna().astype(str).unique().tolist())[:10])),
        "strike_min": saved_chain_df["strike"].min(),
        "strike_max": saved_chain_df["strike"].max(),
        "bid_min": saved_chain_df["bid"].min(),
        "bid_max": saved_chain_df["bid"].max(),
        "ask_min": saved_chain_df["ask"].min(),
        "ask_max": saved_chain_df["ask"].max(),
        "mid_min": saved_chain_df["mid"].min(),
        "mid_max": saved_chain_df["mid"].max(),
    }])
    
    display(saved_summary)
    print()
    print("Saved chain preview:")
    display(saved_chain_df.head(30))

# --------------------------------------------------------------------------------------
# 7) Save manifest
# --------------------------------------------------------------------------------------

manifest_path = AUDIT_DIR / f"02B2_spy_chain_probe_manifest_{timestamp}.json"

manifest = {
    "cell": "2B.2",
    "description": "Inspect existing ThetaData code/schema and test one SPY option-chain pull",
    "created_at": timestamp,
    "snippet_path": str(snippet_path),
    "existing_schema_path": str(existing_schema_path),
    "existing_preview_path": str(existing_preview_path),
    "service_probe_path": str(service_probe_path),
    "expiration_probe_path": str(expiration_probe_path),
    "chain_attempt_path": str(chain_attempt_path),
    "spy_chain_dir": str(SPY_CHAIN_DIR),
    "test_root": TEST_ROOT,
    "test_tenor": TEST_TENOR,
    "test_trade_date": str(test_trade_date.date()),
    "target_expiration_date": str(target_expiration_date.date()),
    "working_base_url": working_base_url,
    "expiration_source": expiration_source,
    "expiration_candidates": expiration_candidates[:50],
    "saved_chain_path": str(saved_chain_path) if saved_chain_path is not None else None,
    "saved_chain_rows": int(len(saved_chain_df)) if saved_chain_path is not None else 0,
}

with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print()
print("=" * 100)
print("Cell 2B.2 complete")
print("=" * 100)
print(f"Manifest: {manifest_path}")

if saved_chain_path is not None:
    print("Result: SPY test chain pull worked. Next cell can build the SPY pull/cache plan.")
else:
    print("Result: SPY test chain pull did not produce a usable saved chain. Review endpoint attempts.")

In [ ]:
# Cell 2B.2 repair — Fast ThetaData SPY service / chain probe only
# Purpose:
#   Continue from the point where Cell 2B.2 appeared to stop at Step 4.
#   This skips the large text/code scan and tests the local ThetaData service directly.

from pathlib import Path
import pandas as pd
import numpy as np
import requests
from io import StringIO
import json
from datetime import datetime

PROJECT_ROOT = Path(r"C:\Users\patri\vrp_project")
AUDIT_DIR = PROJECT_ROOT / "data" / "audit" / "call_sleeve_research"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "call_sleeve_research"
SPY_CHAIN_DIR = PROJECT_ROOT / "data" / "raw" / "thetadata_chains_spy"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
SPY_CHAIN_DIR.mkdir(parents=True, exist_ok=True)

BASE_URLS = [
    "http://127.0.0.1:25510",
    "http://localhost:25510",
]

TEST_ROOT = "SPY"
TEST_TRADE_DATE = pd.Timestamp("2026-07-01")
TEST_EXPIRATION = "20260731"

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 100)
print("Cell 2B.2 repair — Fast ThetaData SPY probe")
print("=" * 100)
print(f"Test root:       {TEST_ROOT}")
print(f"Trade date:      {TEST_TRADE_DATE.date()}")
print(f"Expiration:      {TEST_EXPIRATION}")
print(f"Output dir:      {SPY_CHAIN_DIR}")
print()

session = requests.Session()
session.trust_env = False  # avoids proxy/env weirdness for localhost

def response_to_dataframe(resp):
    text = resp.text.strip()
    if not text:
        return pd.DataFrame()

    try:
        obj = resp.json()
        if isinstance(obj, dict):
            if "response" in obj:
                data = obj.get("response", [])
                header = obj.get("header", {})
                cols = None
                if isinstance(header, dict):
                    for key in ["format", "columns", "fields"]:
                        if key in header and isinstance(header[key], list):
                            cols = header[key]
                            break
                if cols is not None:
                    return pd.DataFrame(data, columns=cols)
                return pd.DataFrame(data)
            return pd.json_normalize(obj)

        if isinstance(obj, list):
            return pd.DataFrame(obj)
    except Exception:
        pass

    try:
        return pd.read_csv(StringIO(text))
    except Exception:
        return pd.DataFrame({"raw_response": text.splitlines()})

def get_df(base_url, endpoint, params=None, timeout=(1, 8)):
    url = base_url.rstrip("/") + endpoint
    params = params or {}

    try:
        resp = session.get(url, params=params, timeout=timeout)
        df = response_to_dataframe(resp)
        return {
            "base_url": base_url,
            "endpoint": endpoint,
            "url": resp.url,
            "params": json.dumps(params),
            "status_code": resp.status_code,
            "ok": resp.ok,
            "rows": len(df),
            "cols": len(df.columns),
            "columns": ", ".join(map(str, df.columns)),
            "text_preview": resp.text[:500],
            "error": None,
            "df": df,
        }
    except Exception as e:
        return {
            "base_url": base_url,
            "endpoint": endpoint,
            "url": url,
            "params": json.dumps(params),
            "status_code": None,
            "ok": False,
            "rows": 0,
            "cols": 0,
            "columns": "",
            "text_preview": "",
            "error": repr(e),
            "df": pd.DataFrame(),
        }

def first_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower:
            return lower[c.lower()]
    return None

def standardize_chain(df, root, exp, quote_date):
    out = df.copy()

    rename = {}
    mapping = {
        "timestamp": ["timestamp", "datetime", "date", "trade_date", "quote_date"],
        "root": ["root", "underlying", "symbol", "ticker"],
        "expiration": ["expiration", "expiry", "expiration_date", "exp"],
        "strike": ["strike"],
        "right": ["right", "option_type", "cp", "call_put", "put_call"],
        "bid": ["bid", "bid_price"],
        "ask": ["ask", "ask_price"],
        "mid": ["mid", "mark"],
        "bid_size": ["bid_size"],
        "ask_size": ["ask_size"],
    }

    for canonical, candidates in mapping.items():
        c = first_col(out, candidates)
        if c is not None:
            rename[c] = canonical

    out = out.rename(columns=rename)

    if "root" not in out.columns:
        out["root"] = root
    if "expiration" not in out.columns:
        out["expiration"] = exp
    if "timestamp" not in out.columns:
        out["timestamp"] = quote_date.strftime("%Y-%m-%dT16:00:00.000")

    for c in ["strike", "bid", "ask"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    if "mid" not in out.columns and {"bid", "ask"}.issubset(out.columns):
        out["mid"] = (out["bid"] + out["ask"]) / 2.0
    elif "mid" in out.columns:
        out["mid"] = pd.to_numeric(out["mid"], errors="coerce")

    out["root"] = out["root"].astype(str).str.upper()
    out["right"] = out["right"].astype(str).str.upper().str[0]
    out["expiration"] = out["expiration"].astype(str).str.replace(".0", "", regex=False)

    return out

# 1) Service probe
probe_rows = []
working_base = None

for base in BASE_URLS:
    for endpoint in ["/v2/system/status", "/v2/list/roots/option", "/"]:
        r = get_df(base, endpoint, timeout=(1, 3))
        probe_rows.append({k: v for k, v in r.items() if k != "df"})
        if r["status_code"] is not None and r["status_code"] < 500 and working_base is None:
            working_base = base

probe_df = pd.DataFrame(probe_rows)
probe_path = AUDIT_DIR / f"02B2_repair_service_probe_{timestamp}.csv"
probe_df.to_csv(probe_path, index=False)

print("=" * 100)
print("Service probe")
print("=" * 100)
display(probe_df[["base_url", "endpoint", "status_code", "ok", "rows", "cols", "columns", "error", "text_preview"]])
print(f"Saved service probe: {probe_path}")
print()

if working_base is None:
    print("No local ThetaData service responded. Start ThetaData Terminal/local API, then rerun this repair cell.")
else:
    print(f"Working base URL candidate: {working_base}")
    print()

# 2) Try expiration list
exp_df = pd.DataFrame()
if working_base is not None:
    exp_attempts = []
    for endpoint in ["/v2/list/expirations", "/v2/list/expirations/option"]:
        r = get_df(working_base, endpoint, params={"root": TEST_ROOT}, timeout=(1, 10))
        exp_attempts.append({k: v for k, v in r.items() if k != "df"})
        if r["rows"] > 0 and exp_df.empty:
            exp_df = r["df"]

    exp_attempts_df = pd.DataFrame(exp_attempts)
    exp_attempts_path = AUDIT_DIR / f"02B2_repair_expiration_attempts_{timestamp}.csv"
    exp_attempts_df.to_csv(exp_attempts_path, index=False)

    print("=" * 100)
    print("Expiration endpoint attempts")
    print("=" * 100)
    display(exp_attempts_df[["endpoint", "status_code", "ok", "rows", "cols", "columns", "error", "text_preview"]])
    print(f"Saved expiration attempts: {exp_attempts_path}")
    print()

# 3) Try quote chain endpoints
saved_path = None
saved = pd.DataFrame()
chain_attempt_rows = []

if working_base is not None:
    trade_yyyymmdd = TEST_TRADE_DATE.strftime("%Y%m%d")

    endpoint_param_sets = [
        (
            "/v2/bulk_hist/option/quote",
            {"root": TEST_ROOT, "exp": TEST_EXPIRATION, "start_date": trade_yyyymmdd, "end_date": trade_yyyymmdd, "ivl": 900000},
        ),
        (
            "/v2/bulk_hist/option/quote",
            {"root": TEST_ROOT, "exp": TEST_EXPIRATION, "start_date": trade_yyyymmdd, "end_date": trade_yyyymmdd},
        ),
        (
            "/v2/bulk_hist/option/eod",
            {"root": TEST_ROOT, "exp": TEST_EXPIRATION, "start_date": trade_yyyymmdd, "end_date": trade_yyyymmdd},
        ),
        (
            "/v2/bulk_hist/option/eod",
            {"root": TEST_ROOT, "exp": TEST_EXPIRATION, "start_date": trade_yyyymmdd, "end_date": trade_yyyymmdd, "ivl": 0},
        ),
    ]

    for endpoint, params in endpoint_param_sets:
        r = get_df(working_base, endpoint, params=params, timeout=(1, 30))
        df = r["df"]

        has_schema = all(
            first_col(df, cols) is not None
            for cols in [
                ["strike"],
                ["right", "option_type", "cp", "call_put", "put_call"],
                ["bid", "bid_price"],
                ["ask", "ask_price"],
            ]
        )

        chain_attempt_rows.append({
            **{k: v for k, v in r.items() if k != "df"},
            "has_usable_schema": has_schema,
        })

        if has_schema and len(df) > 0 and saved_path is None:
            st = standardize_chain(df, TEST_ROOT, TEST_EXPIRATION, TEST_TRADE_DATE)
            valid = st[
                st["strike"].notna()
                & st["right"].isin(["C", "P"])
                & st["bid"].notna()
                & st["ask"].notna()
            ].copy()

            if len(valid) > 0:
                saved_path = SPY_CHAIN_DIR / f"SPY_{trade_yyyymmdd}_{TEST_EXPIRATION}.pkl"
                valid.to_pickle(saved_path)
                saved = valid

chain_attempts_df = pd.DataFrame(chain_attempt_rows)
chain_attempts_path = AUDIT_DIR / f"02B2_repair_chain_attempts_{timestamp}.csv"
chain_attempts_df.to_csv(chain_attempts_path, index=False)

print("=" * 100)
print("SPY chain endpoint attempts")
print("=" * 100)

if chain_attempts_df.empty:
    print("No chain attempts made.")
else:
    display(chain_attempts_df[["endpoint", "params", "status_code", "ok", "rows", "cols", "columns", "has_usable_schema", "error", "text_preview"]])
    print(f"Saved chain attempts: {chain_attempts_path}")

print()

if saved_path is None:
    print("No usable SPY chain saved.")
else:
    print(f"Saved usable SPY chain: {saved_path}")
    print(f"Rows: {len(saved):,}")
    print("Columns:", list(saved.columns))
    print()

    summary = pd.DataFrame([{
        "root_values": ", ".join(sorted(saved["root"].dropna().unique())),
        "right_values": ", ".join(sorted(saved["right"].dropna().unique())),
        "expiration_values": ", ".join(sorted(saved["expiration"].dropna().astype(str).unique())),
        "timestamp_preview": ", ".join(sorted(saved["timestamp"].dropna().astype(str).unique())[:10]),
        "strike_min": saved["strike"].min(),
        "strike_max": saved["strike"].max(),
        "bid_min": saved["bid"].min(),
        "bid_max": saved["bid"].max(),
        "ask_min": saved["ask"].min(),
        "ask_max": saved["ask"].max(),
    }])
    display(summary)
    display(saved.head(30))